# Customer VOC Topic Modeling (LDA)
**Kimberly-Clark AI Labs · 2023**

> **Note:** Output cells have been cleared due to company data confidentiality.
> This notebook demonstrates the methodology and code structure only.

Applied LDA (Latent Dirichlet Allocation) to customer service contact logs (VOC)
to automatically surface recurring complaint topics and support data-driven CX improvements.

### Table of Contents
1. Environment Setup
2. Data Loading & Preprocessing
3. Exploratory Data Analysis
4. Korean Text Tokenization
5. Vectorization & LDA Modeling
6. Topic Visualization (pyLDAvis)
7. Document-Topic Assignment
8. Category-Level Deep Dive

## 1. Environment Setup

In [ ]:
# Install Korean NLP and LDA visualization libraries
# KoNLPy: Korean Natural Language Processing toolkit
# pyLDAvis: Interactive LDA topic visualization
!pip install konlpy pyldavis

In [ ]:
# Install MeCab Korean morphological analyzer (Google Colab)
!git clone https://github.com/SOMJANG/Mecab-ko-for-Google-Colab.git
%cd Mecab-ko-for-Google-Colab
!bash install_mecab-ko_on_colab190912.sh

In [ ]:
# Install Korean font for matplotlib visualizations
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
import re
import io
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import konlpy
from konlpy.tag import Okt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from tqdm import tqdm

# Render plots inline
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# Korean font for matplotlib (required for Korean label rendering)
plt.rc('font', family='NanumBarunGothic')
mpl.rc('axes', unicode_minus=False)

## 2. Data Loading & Preprocessing
Data source: Internal customer service contact logs (VOC), Feb 2022 to Feb 2023.
Columns include: ticket ID, consultation category (Level 1 & 2), and consultation content.

In [ ]:
# Upload data file via Google Colab file picker
from google.colab import files
myfile = files.upload()

In [ ]:
# ── LOAD & PARSE DATA ────────────────────────────────────────────────────
# Parse date from ticket ID prefix (format: YYYYMMDD...)

df = pd.read_csv(io.BytesIO(myfile['CIC_VOC_202202_202302.csv']), encoding='utf8', index_col=0)

df['Date'] = pd.to_datetime(df['TicketID'].apply(
    lambda x: '{}-{}-{}'.format(x[:4], x[4:6], x[6:8])
))
df['Week'] = df['Date'].dt.isocalendar().week
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# Select relevant columns and clean content field
df = df.reset_index(drop=True)
df = df[['TicketID', 'Date', 'Year', 'Month', 'Week',
         'ConsultCategory', 'CategoryL1', 'CategoryL2', 'Content']]

# Drop rows with missing content and remove numeric characters
df['Content'] = df['Content'].dropna()
df['Content'] = df['Content'].replace(r'\d+', '', regex=True)

print(f'Loaded {len(df)} records from {df["Date"].min().date()} to {df["Date"].max().date()}')

## 3. Exploratory Data Analysis

In [ ]:
# ── DISTRIBUTION BY CATEGORY L1 ──────────────────────────────────────────

count = df['CategoryL1'].value_counts()
count.plot.bar(grid=True, figsize=(10, 6), fontsize=13)
plt.title('Ticket Volume by Category (L1)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# ── MONTHLY TREND BY CATEGORY ────────────────────────────────────────────

df['YearMonth'] = df['Date'].dt.strftime('%Y-%m')
grouped = df.groupby(['CategoryL1', 'YearMonth'])['TicketID'].count().reset_index()
grouped['YearMonth'] = pd.to_datetime(grouped['YearMonth'])
grouped = grouped.sort_values(by=['CategoryL1', 'YearMonth'])

plt.figure(figsize=(14, 7))
for category in grouped['CategoryL1'].unique():
    data = grouped[grouped['CategoryL1'] == category]
    plt.plot(data['YearMonth'], data['TicketID'], marker='o', label=category)

plt.xlabel('Month')
plt.ylabel('Ticket Count')
plt.title('Monthly Ticket Volume by Category')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Korean Text Tokenization
Used KoNLPy's Okt (Open Korean Text) morphological analyzer to extract
nouns, verbs, and adjectives from customer consultation text.
Short or repetitive documents (fewer than 3 unique tokens) are filtered out.

In [ ]:
# ── TOKENIZATION FUNCTION ────────────────────────────────────────────────
# Extracts meaningful POS tags (Noun, Verb, Adjective) from Korean text

def tokenize_korean(text):
    """Tokenize Korean text using Okt morphological analyzer.
    Retains nouns, verbs, and adjectives only.
    """
    text = re.sub(r'[^,.?!\w\s]', '', str(text))  # Keep alphanumeric and basic punctuation
    okt = Okt()
    morphs = okt.pos(text)
    words = [
        word for word, pos in morphs
        if pos in ('Noun', 'Verb', 'Adjective')
    ]
    return ' '.join(words)

In [ ]:
# ── TOKENIZE BY CATEGORY ─────────────────────────────────────────────────
# Filter to a specific L1 category before modeling
# Change 'target_category' to analyze a different complaint type

target_category = 'DeliveryComplaint'  # Example: delivery complaints

tokenized_list = [
    tokenize_korean(text)
    for text in df['Content'][df['CategoryL1'] == target_category]
]

# Remove documents with fewer than 3 unique tokens (too sparse for modeling)
tokenized_list = [
    corpus for corpus in tokenized_list
    if len(set(corpus.split())) >= 3
]

print(f'Documents after filtering: {len(tokenized_list)}')

## 5. Vectorization & LDA Modeling

In [ ]:
# ── COUNT VECTORIZATION ──────────────────────────────────────────────────
# Parameters:
#   max_df=0.1: exclude terms appearing in >10% of documents (too common)
#   min_df=2:   exclude terms appearing in fewer than 2 documents (too rare)
#   max_features=1000: vocabulary size cap
#   ngram_range=(1,2): include unigrams and bigrams

STOP_WORDS = ['delivery', 'huggies']  # Domain-specific stopwords

count_vectorizer = CountVectorizer(
    max_df=0.1,
    max_features=1000,
    min_df=2,
    ngram_range=(1, 2),
    stop_words=STOP_WORDS
)

feat_vect = count_vectorizer.fit_transform(tokenized_list)
print(f'Document-term matrix shape: {feat_vect.shape}')

In [ ]:
# ── LDA MODEL ────────────────────────────────────────────────────────────
# n_components: number of topics to extract
# Tune this based on coherence score or domain knowledge

N_TOPICS = 6

lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42)
lda.fit(feat_vect)

In [ ]:
# ── DISPLAY TOP WORDS PER TOPIC ──────────────────────────────────────────

def display_topics(model, feature_names, num_top_words=10):
    """Print the top N words for each discovered topic."""
    for topic_index, topic in enumerate(model.components_):
        top_word_indices = topic.argsort()[::-1][:num_top_words]
        top_words = ' | '.join([feature_names[i] for i in top_word_indices])
        print(f'Topic #{topic_index}: {top_words}')

feature_names = count_vectorizer.get_feature_names_out()
display_topics(lda, feature_names, num_top_words=10)

## 6. Topic Visualization (pyLDAvis)
Interactive visualization showing topic distribution and inter-topic distances.
Larger circles indicate higher topic prevalence; closer circles indicate topic overlap.

In [ ]:
import pyLDAvis.lda_model

pyLDAvis.enable_notebook()
vis = pyLDAvis.lda_model.prepare(lda, feat_vect, count_vectorizer)
pyLDAvis.display(vis)

## 7. Document-Topic Assignment
Each document is assigned to the topic with the highest posterior probability.

In [ ]:
# ── ASSIGN DOCUMENTS TO TOPICS ───────────────────────────────────────────

doc_topic_matrix = lda.transform(feat_vect)

doc_topic_records = [
    {'Doc_Num': n,
     'Topic': doc_topic_matrix[n].argmax(),
     'Probability': doc_topic_matrix[n].max()}
    for n in range(doc_topic_matrix.shape[0])
]

doc_topic_df = pd.DataFrame(doc_topic_records)

# Join with original dataframe to inspect content per topic
doc_topic_df = doc_topic_df.join(
    df[df['CategoryL1'] == target_category].reset_index(drop=True)
)

print('Document count per topic:')
print(doc_topic_df.groupby('Topic')[['Doc_Num']].count())

In [ ]:
# ── SAMPLE TOP DOCUMENTS PER TOPIC ──────────────────────────────────────
# Display the highest-confidence document for each topic

for topic in sorted(doc_topic_df['Topic'].unique()):
    print(f'Topic #{topic} {"-" * 40}')
    top_docs = doc_topic_df[doc_topic_df['Topic'] == topic].sort_values(
        by='Probability', ascending=False
    )
    for i in range(min(3, len(top_docs))):
        print(f'  [{i+1}] {top_docs["Content"].iloc[i][:100]}')
    print()

## 8. Category-Level Deep Dive
Drill down into specific complaint categories to analyze sub-type distributions and trends.

In [ ]:
def analyze_category(df, category_name, date_col='Date', level2_col='CategoryL2', ticket_col='TicketID'):
    """Plot L2 sub-category distribution and monthly trend for a given L1 category."""
    subset = df[df['CategoryL1'] == category_name].copy()
    subset['YearMonth'] = subset[date_col].dt.strftime('%Y-%m')

    # Sub-category distribution
    print(f'--- {category_name}: L2 Sub-category Counts ---')
    print(subset[level2_col].value_counts())

    # Monthly trend by L2 sub-category
    grouped = subset.groupby([level2_col, 'YearMonth'])[ticket_col].count().reset_index()
    grouped['YearMonth'] = pd.to_datetime(grouped['YearMonth'])
    grouped = grouped.sort_values(by=[level2_col, 'YearMonth'])

    plt.figure(figsize=(14, 6))
    for subcategory in grouped[level2_col].unique():
        data = grouped[grouped[level2_col] == subcategory]
        plt.plot(data['YearMonth'], data[ticket_col], marker='o', label=subcategory)

    plt.xlabel('Month')
    plt.ylabel('Ticket Count')
    plt.title(f'Monthly Trend: {category_name} Sub-categories')
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Analyze specific complaint categories
# Adjust category names to match actual L1 values in your data

for cat in ['DeliveryComplaint', 'EventComplaint', 'CancellationRefund']:
    analyze_category(df, cat)